# Encoder Comparison: RoBERTa + MiniLM Embeddings (N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**Purpose:** add two more text encoders alongside the BERT results you already have,
to show the quantum kernel finding is not a BERT-specific artifact.

**This notebook is parametrized.** Set `DATASET` and `ENCODER` at the top, run it,
then change them and run again. Four configurations total:
- RoBERTa  + SST-2
- RoBERTa  + AG News
- MiniLM   + SST-2
- MiniLM   + AG News

**Encoder notes:**
- RoBERTa (roberta-base): 768-dim, uses first-token pooling like BERT
- MiniLM (all-MiniLM-L6-v2): 384-dim sentence-transformer, uses mean pooling
  (handled automatically by the sentence-transformers library)

**Runtime:** ~8 min per configuration on Colab T4

In [ ]:
!pip install datasets transformers torch sentence-transformers scikit-learn numpy --quiet

import numpy as np
import time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Choose dataset and encoder — CHANGE THESE EACH RUN

Run this notebook 4 times, once per combination.

In [ ]:
# ============================================
# SET THESE TWO VALUES, THEN RUN THE WHOLE NOTEBOOK
DATASET = 'sst2'        # 'sst2' or 'agnews'
ENCODER = 'roberta'     # 'roberta' or 'minilm'
# ============================================

# Derived config
CONFIG = {
    'sst2':   {'hf_path': 'nyu-mll/glue', 'hf_name': 'sst2', 'text_col': 'sentence',
               'split_test': 'validation', 'n_classes': 2, 'metric': 'binary'},
    'agnews': {'hf_path': 'fancyzhx/ag_news', 'hf_name': None, 'text_col': 'text',
               'split_test': 'test', 'n_classes': 4, 'metric': 'macro'},
}[DATASET]

ENCODER_CFG = {
    'roberta': {'model': 'roberta-base', 'dim': 768, 'type': 'hf'},
    'minilm':  {'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dim': 384, 'type': 'st'},
}[ENCODER]

TAG = f"{DATASET}_{ENCODER}"   # used in all filenames
print(f"Configuration: {DATASET.upper()} + {ENCODER.upper()}")
print(f"  Encoder    : {ENCODER_CFG['model']} ({ENCODER_CFG['dim']}-dim)")
print(f"  Classes    : {CONFIG['n_classes']}")
print(f"  Metric     : {CONFIG['metric']} F1")
print(f"  File tag   : {TAG}")

---
## Step 1: Storage + checkpoint helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

print("Helpers ready.")

---
## Step 2: Load dataset

In [ ]:
from datasets import load_dataset

print(f"Loading {DATASET}...")
if CONFIG['hf_name']:
    dataset = load_dataset(CONFIG['hf_path'], CONFIG['hf_name'])
else:
    dataset = load_dataset(CONFIG['hf_path'])

N_TRAIN_EXTRACT = 1100
N_TEST_EXTRACT  = 250

train_data = dataset['train'].shuffle(seed=42).select(range(N_TRAIN_EXTRACT))
test_data  = dataset[CONFIG['split_test']].shuffle(seed=42).select(range(N_TEST_EXTRACT))

train_texts  = [x[CONFIG['text_col']] for x in train_data]
train_labels = np.array([x['label'] for x in train_data])
test_texts   = [x[CONFIG['text_col']] for x in test_data]
test_labels  = np.array([x['label'] for x in test_data])

print(f"Train {len(train_texts)} | Test {len(test_texts)}")
print(f"Classes: {sorted(set(train_labels))}")
for i in range(2):
    print(f"  [{train_labels[i]}] {train_texts[i][:70]}...")

---
## Step 3: Extract embeddings — checkpoint after

Handles both encoder types: HuggingFace [CLS] pooling (RoBERTa) and
sentence-transformers mean pooling (MiniLM).

In [ ]:
print(f"Extracting {ENCODER} embeddings...")
t0 = time.time()

if ENCODER_CFG['type'] == 'hf':
    # RoBERTa — first-token (CLS-style) pooling, same as BERT
    from transformers import AutoTokenizer, AutoModel
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_CFG['model'])
    model     = AutoModel.from_pretrained(ENCODER_CFG['model']).to(DEVICE)
    model.eval()

    def extract(texts, batch_size=32):
        out = []
        nb = (len(texts) + batch_size - 1) // batch_size
        for i in range(0, len(texts), batch_size):
            batch  = texts[i:i+batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True,
                               max_length=128, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                o = model(**inputs)
            out.append(o.last_hidden_state[:, 0, :].cpu().numpy())  # first token
            if (i//batch_size+1) % 5 == 0: print(f"  batch {i//batch_size+1}/{nb}")
        return np.vstack(out)

    train_emb = extract(train_texts)
    test_emb  = extract(test_texts)

elif ENCODER_CFG['type'] == 'st':
    # MiniLM — sentence-transformers handles mean pooling automatically
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(ENCODER_CFG['model'], device=DEVICE)
    train_emb = model.encode(train_texts, batch_size=64, show_progress_bar=True,
                             convert_to_numpy=True)
    test_emb  = model.encode(test_texts, batch_size=64, show_progress_bar=True,
                             convert_to_numpy=True)

print(f"\nDone in {time.time()-t0:.0f}s")
print(f"Train emb: {train_emb.shape}  (expected {ENCODER_CFG['dim']}-dim)")
print(f"Test  emb: {test_emb.shape}")
assert train_emb.shape[1] == ENCODER_CFG['dim'], "Unexpected embedding dimension"

print("\nCheckpoint 1 — raw embeddings:")
checkpoint(f'{TAG}_raw.npz',
           lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                              train_labels=train_labels, test_labels=test_labels))

---
## Step 4: PCA + normalize — checkpoint after

In [ ]:
N_QUBITS = 8
pca_dims = [4, 8, 12, 16]
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in pca_dims}

print(f"PCA variance explained ({ENCODER_CFG['dim']}-dim -> reduced):")
for n in pca_dims:
    print(f"  {n:>2} dims -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")

pca_8 = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_test_pca  = pca_8.transform(test_emb)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled  = np.clip(scaler.transform(X_test_pca), 0, np.pi)

print(f"\nTrain range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")

print("\nCheckpoint 2 — PCA + scaler:")
checkpoint(f'{TAG}_pca_scaler.pkl',
           lambda p: pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS,
                                  'dataset': DATASET, 'encoder': ENCODER,
                                  'n_classes': CONFIG['n_classes']}, open(p,'wb')))

---
## Step 5: Classical baselines on real 1000 / 200 split

In [ ]:
N_TRAIN_SVM, N_TEST_SVM = 1000, 200
assert len(X_train_scaled) >= N_TRAIN_SVM
assert len(X_test_scaled)  >= N_TEST_SVM

idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_test_scaled),  N_TEST_SVM,  replace=False)
X_tr, y_tr = X_train_scaled[idx_tr], train_labels[idx_tr]
X_te, y_te = X_test_scaled[idx_te],  test_labels[idx_te]

avg = 'binary' if CONFIG['metric'] == 'binary' else 'macro'
def score_f1(yt, yp): return float(f1_score(yt, yp, average=avg))

print(f"Train {X_tr.shape} | Test {X_te.shape} | metric={avg} F1")
print()

classifiers = {
    'Linear':     SVC(kernel='linear', C=1.0, random_state=42),
    'RBF':        SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'Polynomial': SVC(kernel='poly',   C=1.0, degree=3, coef0=1, random_state=42),
}
results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'F1':>10}")
print("-"*36)
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    yp = clf.predict(X_te)
    results[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': score_f1(y_te, yp)}
    print(f"{name:<14} {results[name]['accuracy']:>10.4f} {results[name]['f1']:>10.4f}")

print("\nCheckpoint 3 — baselines:")
checkpoint(f'{TAG}_baseline_results.pkl', lambda p: pickle.dump(results, open(p,'wb')))

---
## Step 6: Save final embeddings — shape verified

In [ ]:
def save_final(p):
    np.savez(p, X_train=X_train_scaled, X_test=X_test_scaled,
             y_train=train_labels, y_test=test_labels,
             X_tr_subset=X_tr, X_te_subset=X_te,
             y_tr_subset=y_tr, y_te_subset=y_te)

print("Checkpoint 4 — final embeddings:")
checkpoint(f'{TAG}_embeddings.npz', save_final)

loaded = np.load(f'{TAG}_embeddings.npz')
print(f"\n  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8)
assert loaded['X_te_subset'].shape == (200, 8)
print("\n  VERIFIED.")

meta = {'dataset': DATASET, 'encoder': ENCODER, 'encoder_model': ENCODER_CFG['model'],
        'embedding_dim': ENCODER_CFG['dim'], 'n_classes': CONFIG['n_classes'],
        'metric': CONFIG['metric'], 'n_train': 1000, 'n_test': 200, 'baselines': results}
checkpoint(f'{TAG}_meta.json', lambda p: json.dump(meta, open(p,'w'), indent=2))

print(f"\nFiles for {TAG}:")
for f in sorted(os.listdir(SAVE_DIR)):
    if f.startswith(TAG):
        print(f"  {f}")

---
## Done — now change config and re-run

### To complete all four configurations:
1. RoBERTa + SST-2  (DATASET='sst2',   ENCODER='roberta')
2. RoBERTa + AG News (DATASET='agnews', ENCODER='roberta')
3. MiniLM  + SST-2  (DATASET='sst2',   ENCODER='minilm')
4. MiniLM  + AG News (DATASET='agnews', ENCODER='minilm')

Change the two values in Step 0, then Runtime > Restart and run all.
Each produces files tagged with the dataset and encoder, so nothing overwrites
your existing BERT results.

### After all four embedding configs are done:
Run the quantum kernel (Week 3-style) and ablations (Week 4-style) for each.
The quantum notebooks just need to load `{dataset}_{encoder}_embeddings.npz`
instead of the BERT files. The metric (binary vs macro F1) is recorded in
each meta.json so you stay consistent.

### Filename convention:
Everything is tagged: sst2_roberta_embeddings.npz, agnews_minilm_baseline_results.pkl,
and so on. Your BERT files (embeddings_week2.npz, ag_news_embeddings.npz) are untouched.